# COU 07506 Digital Signal Processing — Week 01: Lab 0: Setting Up the DSP Laboratory

Name: Renatus Rweyemamu Leopard  
Registration No.: 240647461275  
Class:BEnN24COE 4 
Date: 22/8/2026  

## Pre-lab

**Question:** List three devices you used today that contain DSP, and for each name one signal it processes.

| Device | Signal it processes |
|---|---|
| Smartphone | Speech picked up by the microphone (digitised, compressed and sent over the network) |
| Wi-Fi router / mobile data modem | The modulated radio signal (filtered, demodulated and error-corrected) |
| Wireless earphones or a laptop speaker | The digital music/audio stream (equalised, filtered and converted back to analog sound) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
plt.rcParams["figure.figsize"] = (9, 3.5)

## Task 1.1 — Your first notebook

Elementary sequences: unit impulse, unit step, unit ramp and a real exponential.

In [ ]:
n = np.arange(-5, 11)
delta = (n == 0).astype(float)
step = (n >= 0).astype(float)
ramp = n * step
expo = 0.8**n * step

fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
for ax, x, name in zip(axes, [delta, step, ramp, expo],
                       ["δ(n)", "u(n)", "r(n)", "(0.8)^n u(n)"]):
    ax.stem(n, x, basefmt="k-")
    ax.set_title(name)
    ax.set_xlabel("n (samples)")
    ax.set_ylabel("amplitude")
plt.tight_layout()
plt.show()

## Task 1.2 — Change the parameters

**Part 1:** $(a)^n u(n)$ for $a = 0.5$, $a = -0.8$ and $a = 1.1$.

In [ ]:
n = np.arange(0, 21)
fig, axes = plt.subplots(1, 3, figsize=(13, 2.8))
for ax, a in zip(axes, [0.5, -0.8, 1.1]):
    ax.stem(n, a**n, basefmt="k-")
    ax.set_title(f"a = {a}")
    ax.set_xlabel("n (samples)")
    ax.set_ylabel("$a^n u(n)$")
plt.tight_layout()
plt.show()

**Part 2:** $\cos(2\pi n/8)$ and $\cos(2\pi n/7.5)$ for $0 \le n \le 40$.

In [ ]:
n = np.arange(0, 41)
x1 = np.cos(2*np.pi*n/8)
x2 = np.cos(2*np.pi*n/7.5)

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
axes[0].stem(n, x1, basefmt="k-")
axes[0].set_title("cos(2πn/8)")
axes[1].stem(n, x2, basefmt="k-")
axes[1].set_title("cos(2πn/7.5)")
for ax in axes:
    ax.set_xlabel("n (samples)")
    ax.set_ylabel("amplitude")
plt.tight_layout()
plt.show()

def smallest_period(x, max_p=30):
    for p in range(1, max_p + 1):
        if np.allclose(x[p:], x[:-p]):
            return p
    return None

print("smallest repeat period of cos(2πn/8)  :", smallest_period(x1), "samples")
print("smallest repeat period of cos(2πn/7.5):", smallest_period(x2), "samples")

**Observation.** $\cos(2\pi n/8)$ repeats exactly every 8 samples, so it looks periodic straight away.
$\cos(2\pi n/7.5)$ does **not** repeat after 7.5 samples, because a discrete-time signal can only repeat after a whole number of samples.
It does repeat exactly after 15 samples (two full cycles), as the printout confirms. So both are periodic, but the second one has a longer period than its cycle length suggests.
This is the idea of periodicity in discrete time (frequency must be a rational multiple of $2\pi$) that is explained in Week 3.

## Task 1.3 — Working with audio

If the file `my_voice.wav` is in the same folder as this notebook, it is used. Otherwise the fallback 440 Hz test tone is used so the notebook still runs from top to bottom.

In [ ]:
import os

fname = "my_voice.wav"
using_fallback = not os.path.exists(fname)
if not using_fallback:
    fs, x = wavfile.read(fname)
    if x.ndim > 1:
        x = x.mean(axis=1)
    x = x / np.max(np.abs(x))
else:
    fs = 16000
    t = np.arange(0, 2, 1/fs)
    x = 0.5*np.sin(2*np.pi*440*t) * np.exp(-t)

t = np.arange(len(x)) / fs
print("source:", "fallback test tone" if using_fallback else fname)
print(f"fs = {fs} Hz, duration = {len(x)/fs:.2f} s, samples = {len(x)}")

plt.plot(t, x)
plt.xlabel("time (s)")
plt.ylabel("amplitude")
plt.title("My recording")
plt.show()

In [ ]:
from IPython.display import Audio
Audio(x, rate=fs)

## Task 1.4 — The same in Octave (optional)

Not attempted (optional). In Octave the equivalent of Task 1.1 would be `n = -5:10; stem(n, double(n>=0))`.
Differences from Python: ranges use `start:stop` and include the end point, indexing starts at 1, and there is no `np.` prefix.

## Task 1.5 — Repository and submission

- Git repository: `COU07506-labs`
- Repository link: ......................................................
- This notebook was committed as `COU07506_Week01_<RegNo>.ipynb` and uploaded to DIT-eLMS.

## Checkpoint 1

Plots of Tasks 1.1–1.3 and the printed sampling rate and duration (above) are ready to show the instructor.

## Analysis questions

In [ ]:
for rate in [fs, 44100, 48000]:
    samples = 60 * rate
    print(f"fs = {rate:5d} Hz -> {samples:>9,d} numbers per minute (mono), "
          f"{samples*2/1e6:.2f} MB at 16 bits per sample")

**1. Sampling rate and storage for one minute.**
The rate of my recording is the value printed in Task 1.3 (for the fallback tone it is 16 000 Hz; phone recordings are usually 44 100 Hz or 48 000 Hz).
One minute in mono needs $60 \times f_s$ numbers: 960 000 at 16 kHz, 2 646 000 at 44.1 kHz and 2 880 000 at 48 kHz (1.9 MB at 16 kHz and 5.3 to 5.8 MB at 44.1 to 48 kHz, at 16 bits per sample).

**2. Why a stem plot for $x(n)$?**
A discrete-time signal exists only at integer values of $n$. There is no value between samples, so drawing a continuous line would suggest values that do not exist.
A stem plot shows exactly one value at each integer index and nothing in between.

**3. The $a = 1.1$ exponential as an impulse response.**
It grows without limit, so $\sum_n |h(n)| = \infty$. This violates **BIBO stability**: a bounded input (for example a single impulse) would produce an output that grows without bound.
(It is causal, since $h(n)=0$ for $n<0$, but it is not stable.)

## Pre-lab, tasks and deliverables checklist

- [x] Pre-lab answered
- [x] Tasks 1.1–1.3 completed
- [x] Analysis questions answered
- [ ] Mini-project group formed (3–5 students) and two topics listed in the eLMS forum

## AI-use statement

**AI tools used:** Claude (Anthropic)  
**Used for:** helping to organise the notebook, check the code of Tasks 1.1–1.3 and draft the answers to the analysis questions.  
**How I verified the output:** compared the plots with the expected figure in the manual; checked the periods of the two cosines numerically with `smallest_period`; checked the storage numbers by hand ($60 \times f_s$).  
I confirm that I understand and can explain all submitted code.